# LeetCode 3470 — Permutations IV bằng Python

- Link: https://leetcode.com/problems/permutations-iv/
- Độ khó: **Hard**
- Chủ đề: **Tìm hoán vị thứ k theo thứ tự từ điển** bằng **đếm số cách hoàn thành** (tổ hợp: `o! · e!`), ràng buộc **xen kẽ chẵn/lẻ**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def permute(self, n: int, k: int) -> List[int]:
```

## 1. Mục tiêu học

1. Nắm khuôn "phần tử thứ `k` theo thứ tự từ điển": thử từng ứng viên tăng dần, **đếm số đối tượng** bắt đầu bằng tiền tố đó, trừ dần `k`.
2. Đếm số hoán vị xen kẽ hoàn thành được khi còn `o` số lẻ, `e` số chẵn và biết tính chẵn lẻ bắt buộc của phần tử kế tiếp: `o! · e!` nếu khả thi, ngược lại `0`.
3. Hiểu điều kiện khả thi: dãy xen kẽ bắt đầu bằng lẻ cần `o = e` hoặc `o = e + 1`.
4. Tăng tốc: mọi ứng viên **cùng tính chẵn lẻ** cho **cùng số cách** ⇒ dùng phép chia để nhảy thẳng.
5. Xử lý `k` vượt tổng số (trả `[]`), `k ≤ 10^15` trong khi số hoán vị có thể rất lớn.

## 2. Đề bài & ví dụ

Cho `n` và `k`. Một **hoán vị xen kẽ** của `1..n` là hoán vị mà hai phần tử kề nhau **không cùng tính chẵn lẻ**. Trả về hoán vị xen kẽ thứ `k` (đánh số từ 1) theo thứ tự từ điển; nếu không đủ `k` hoán vị, trả `[]`.

**Ví dụ 1:** `n = 4, k = 6` → `[3, 4, 1, 2]`.

**Ví dụ 2:** `n = 3, k = 2` → `[3, 2, 1]`.

**Ví dụ 3:** `n = 2, k = 3` → `[]`.

**Ràng buộc:** `1 ≤ n ≤ 100`, `1 ≤ k ≤ 10^15`.

In [1]:
from typing import List
import itertools, math

examples = [(4, 6, [3, 4, 1, 2]), (3, 2, [3, 2, 1]), (2, 3, [])]

def is_alternating(p):
    return all((p[i] - p[i + 1]) % 2 == 1 for i in range(len(p) - 1))

for n, k, exp in examples:
    print(n, k, '-> kỳ vọng', exp)

4 6 -> kỳ vọng [3, 4, 1, 2]
3 2 -> kỳ vọng [3, 2, 1]
2 3 -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Liệt kê hoán vị theo thứ tự từ điển

`itertools.permutations(range(1, n + 1))` sinh hoán vị **theo thứ tự từ điển** khi đầu vào đã sắp tăng.

In [2]:
alts = [p for p in itertools.permutations(range(1, 5)) if is_alternating(p)]
for i, p in enumerate(alts, 1):
    print(i, p)

1 (1, 2, 3, 4)
2 (1, 4, 3, 2)
3 (2, 1, 4, 3)
4 (2, 3, 4, 1)
5 (3, 2, 1, 4)
6 (3, 4, 1, 2)
7 (4, 1, 2, 3)
8 (4, 3, 2, 1)


### 3.2. Đếm hoán vị xen kẽ

Với `o = ⌈n/2⌉` số lẻ và `e = ⌊n/2⌋` số chẵn, một **mẫu chẵn lẻ** xen kẽ cố định (ví dụ L C L C …) được lấp bằng cách xếp các số lẻ vào ô lẻ (`o!` cách) và số chẵn vào ô chẵn (`e!` cách).

- `n` lẻ: chỉ mẫu bắt đầu bằng **lẻ** khả thi ⇒ `o! · e!`.
- `n` chẵn: hai mẫu ⇒ `2 · o! · e!`.

In [3]:
def total_alternating(n):
    o, e = (n + 1) // 2, n // 2
    return math.factorial(o) * math.factorial(e) * (1 if n % 2 else 2)

for n in range(1, 9):
    cnt = sum(1 for p in itertools.permutations(range(1, n + 1)) if is_alternating(p))
    print(n, cnt, total_alternating(n))
print('n = 100:', total_alternating(100), '(lớn hơn 10^15 rất nhiều)')

1 1 1
2 2 2
3 2 2
4 8 8
5 12 12
6 72 72
7 144 144
8 1152 1152
n = 100: 1850034130565015838026941446471767364698973614843803975412278542037621141434720868884766426280896430604288000000000000000000000000 (lớn hơn 10^15 rất nhiều)


### 3.3. Số cách hoàn thành khi biết phần tử kế tiếp phải lẻ/chẵn

Còn `o` lẻ, `e` chẵn, phần tử kế tiếp phải **lẻ** ⇒ mẫu L C L C … dài `o + e`, cần `o = e` hoặc `o = e + 1`. Tương tự cho chẵn. Khả thi ⇒ `o! · e!`, không ⇒ `0`. Hết số (`o + e = 0`) ⇒ `1` cách (dãy rỗng).

In [4]:
def ways(o, e, need_odd):
    if o + e == 0:
        return 1
    ok = (o == e or o == e + 1) if need_odd else (e == o or e == o + 1)
    return math.factorial(o) * math.factorial(e) if ok else 0

print(ways(2, 2, True), ways(2, 2, False), ways(2, 1, True), ways(2, 1, False), ways(0, 0, True))

4 4 2 0 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`n = 4, k = 6`): `o = e = 2`. Vị trí đầu tiên thử `1, 2, 3, 4`:

- Đặt `1`: còn `o = 1, e = 2`, kế tiếp phải chẵn: `e = o + 1` ✓ ⇒ `1! · 2! = 2` hoán vị. `k = 6 > 2` ⇒ `k = 4`.
- Đặt `2`: còn `o = 2, e = 1`, kế tiếp lẻ ⇒ `2` hoán vị. `k = 4 > 2` ⇒ `k = 2`.
- Đặt `3`: còn `o = 1, e = 2` ⇒ `2` hoán vị. `k = 2 ≤ 2` ⇒ chọn `3`.

Vị trí 2 (phải chẵn): thử `2`: còn `o = 1, e = 1`, kế tiếp lẻ ⇒ `1`; `k = 2 > 1` ⇒ `k = 1`. Thử `4` ⇒ `1` ⇒ chọn `4`. Phần còn lại buộc: `1`, `2` ⇒ `[3, 4, 1, 2]`.

In [5]:
print('các hoán vị bắt đầu bằng 1, 2, 3:', [sum(1 for p in alts if p[0] == d) for d in (1, 2, 3)])

các hoán vị bắt đầu bằng 1, 2, 3: [2, 2, 2]


## 5. Cách 1 — Brute Force: liệt kê theo thứ tự từ điển

In [6]:
def permute_brute(n, k):
    c = 0
    for p in itertools.permutations(range(1, n + 1)):
        if is_alternating(p):
            c += 1
            if c == k:
                return list(p)
    return []

for n, k, exp in examples:
    print(permute_brute(n, k), '| kỳ vọng', exp)

[3, 4, 1, 2] | kỳ vọng [3, 4, 1, 2]
[3, 2, 1] | kỳ vọng [3, 2, 1]
[] | kỳ vọng []


**Độ phức tạp:** `O(n! · n)` — chỉ dùng cho `n ≤ 9`.

## 6. Ý tưởng thuật toán / bước đột phá

**Khuôn phần tử thứ `k`:** dựng kết quả từng vị trí. Tại mỗi vị trí, duyệt ứng viên chưa dùng theo thứ tự tăng (và đúng tính chẵn lẻ bắt buộc), tính `w` = số hoán vị xen kẽ có tiền tố hiện tại nối thêm ứng viên. Nếu `k > w` thì bỏ qua cả khối, `k −= w`; ngược lại chọn ứng viên.

**Đột phá đếm:** `w` chỉ phụ thuộc `(o, e)` còn lại và tính chẵn lẻ bắt buộc kế tiếp ⇒ `ways(o, e, need)` = `o! · e!` hoặc `0`.

**Đột phá tốc độ:** mọi ứng viên cùng tính chẵn lẻ cho **cùng** `w` ⇒ chỉ số trong nhóm là `(k − 1) // w`, không cần trừ từng cái.

**Lưu ý `k` lớn:** `o! · e!` có thể vượt xa `10^15` — Python dùng số nguyên lớn nên không lo tràn (ngôn ngữ khác cần "kẹp" ở `10^15 + 1`).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đếm bằng quy hoạch động (không cần công thức giai thừa)

`f(o, e, need)`: số cách xếp hết `o` số lẻ và `e` số chẵn khi phần tử đầu phải có tính `need`. `f(o, e, lẻ) = o · f(o − 1, e, chẵn)` (chọn **số lẻ cụ thể** nào: `o` cách), tương tự cho chẵn; `f(0, 0, ·) = 1`.

In [7]:
from functools import lru_cache

@lru_cache(maxsize=None)
def f_dp(o, e, need_odd):
    if o + e == 0:
        return 1
    if need_odd:
        return o * f_dp(o - 1, e, False) if o else 0
    return e * f_dp(o, e - 1, True) if e else 0

def permute_dp(n, k):
    used = [False] * (n + 1)
    o, e = (n + 1) // 2, n // 2
    res, last = [], None
    for _ in range(n):
        for c in range(1, n + 1):
            if used[c] or (last is not None and c % 2 == last):
                continue
            o2, e2 = (o - 1, e) if c % 2 else (o, e - 1)
            w = f_dp(o2, e2, c % 2 == 0)           # kế tiếp phải khác tính của c
            if k > w:
                k -= w
            else:
                used[c] = True; res.append(c); o, e, last = o2, e2, c % 2
                break
        else:
            return []
    return res

assert all(f_dp(o, e, nd) == ways(o, e, nd) for o in range(8) for e in range(8) for nd in (True, False))
for n, k, exp in examples:
    print(permute_dp(n, k), '| kỳ vọng', exp)

[3, 4, 1, 2] | kỳ vọng [3, 4, 1, 2]
[3, 2, 1] | kỳ vọng [3, 2, 1]
[] | kỳ vọng []


### 7.2. Cách 3 — Công thức `o! · e!`, trừ từng ứng viên

In [8]:
def permute_formula(n, k):
    used = [False] * (n + 1)
    o, e = (n + 1) // 2, n // 2
    res, last = [], None
    for _ in range(n):
        for c in range(1, n + 1):
            if used[c] or (last is not None and c % 2 == last):
                continue
            o2, e2 = (o - 1, e) if c % 2 else (o, e - 1)
            w = ways(o2, e2, c % 2 == 0)
            if k > w:
                k -= w
            else:
                used[c] = True; res.append(c); o, e, last = o2, e2, c % 2
                break
        else:
            return []
    return res

for n, k, exp in examples:
    print(permute_formula(n, k), '| kỳ vọng', exp)

[3, 4, 1, 2] | kỳ vọng [3, 4, 1, 2]
[3, 2, 1] | kỳ vọng [3, 2, 1]
[] | kỳ vọng []


### 7.3. Cách 4 — Nhảy theo nhóm chẵn lẻ bằng phép chia

Ở vị trí đầu tiên (khi `n` chẵn), có hai nhóm ứng viên: **lẻ** rồi **chẵn**, xen lẫn theo thứ tự giá trị. Vì `1 < 2 < 3 < …`, thứ tự từ điển xen kẽ nhóm lẻ và chẵn — nên ở vị trí đầu vẫn phải duyệt theo giá trị; nhưng mọi ứng viên có **cùng** `w` (khi `n` chẵn, `o = e` nên đặt lẻ hay chẵn đều còn lại cấu hình đối xứng, `w` bằng nhau) ⇒ chỉ số `(k − 1) // w` trong danh sách **tất cả** ứng viên. Từ vị trí thứ hai trở đi, ứng viên chỉ có một tính chẵn lẻ ⇒ cũng một `w` chung.

In [9]:
def permute_jump(n, k):
    o, e = (n + 1) // 2, n // 2
    remaining_odd = list(range(1, n + 1, 2))
    remaining_even = list(range(2, n + 1, 2))
    res, last = [], None
    for _ in range(n):
        if last is None:
            cands = sorted(remaining_odd + remaining_even) if n % 2 == 0 else remaining_odd[:]
        else:
            cands = remaining_even if last == 1 else remaining_odd
        c0 = cands[0]
        o2, e2 = (o - 1, e) if c0 % 2 else (o, e - 1)
        w = ways(o2, e2, c0 % 2 == 0)              # như nhau cho mọi ứng viên (xem giải thích)
        if w == 0:
            return []
        idx = (k - 1) // w
        if idx >= len(cands):
            return []
        c = cands[idx]
        k -= idx * w
        (remaining_odd if c % 2 else remaining_even).remove(c)
        o, e = (o - 1, e) if c % 2 else (o, e - 1)
        res.append(c); last = c % 2
    return res

for n, k, exp in examples:
    print(permute_jump(n, k), '| kỳ vọng', exp)

[3, 4, 1, 2] | kỳ vọng [3, 4, 1, 2]
[3, 2, 1] | kỳ vọng [3, 2, 1]
[] | kỳ vọng []


Vì sao ở vị trí đầu với `n` chẵn mọi ứng viên có cùng `w`: đặt số lẻ ⇒ còn `(o − 1, e)` cần chẵn tiếp theo: `e = (o − 1) + 1` ✓ ⇒ `(o−1)! · e!`. Đặt số chẵn ⇒ còn `(o, e − 1)` cần lẻ: `o = (e − 1) + 1` ✓ ⇒ `o! · (e−1)!`. Vì `o = e` nên hai số bằng nhau.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1: in mỗi ứng viên, số hoán vị trong khối và `k` sau khi trừ.

In [10]:
def trace_formula(n, k):
    used = [False] * (n + 1)
    o, e = (n + 1) // 2, n // 2
    res, last = [], None
    for pos in range(n):
        for c in range(1, n + 1):
            if used[c] or (last is not None and c % 2 == last):
                continue
            o2, e2 = (o - 1, e) if c % 2 else (o, e - 1)
            w = ways(o2, e2, c % 2 == 0)
            if k > w:
                print(f"vị trí {pos}: thử {c}: khối có {w} hoán vị, k={k} > {w} -> bỏ qua, k={k - w}")
                k -= w
            else:
                print(f"vị trí {pos}: thử {c}: khối có {w} hoán vị, k={k} <= {w} -> CHỌN {c}")
                used[c] = True; res.append(c); o, e, last = o2, e2, c % 2
                break
        else:
            print('không đủ hoán vị'); return []
    print('kết quả:', res)
    return res

trace_formula(4, 6)

vị trí 0: thử 1: khối có 2 hoán vị, k=6 > 2 -> bỏ qua, k=4
vị trí 0: thử 2: khối có 2 hoán vị, k=4 > 2 -> bỏ qua, k=2
vị trí 0: thử 3: khối có 2 hoán vị, k=2 <= 2 -> CHỌN 3
vị trí 1: thử 2: khối có 1 hoán vị, k=2 > 1 -> bỏ qua, k=1
vị trí 1: thử 4: khối có 1 hoán vị, k=1 <= 1 -> CHỌN 4
vị trí 2: thử 1: khối có 1 hoán vị, k=1 <= 1 -> CHỌN 1
vị trí 3: thử 2: khối có 1 hoán vị, k=1 <= 1 -> CHỌN 2
kết quả: [3, 4, 1, 2]


[3, 4, 1, 2]

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def permute(self, n: int, k: int) -> List[int]:
        fact = [1] * (n + 1)
        for i in range(1, n + 1):
            fact[i] = fact[i - 1] * i

        def ways(o, e, need_odd):
            if o + e == 0:
                return 1
            if need_odd:
                ok = o == e or o == e + 1
            else:
                ok = e == o or e == o + 1
            return fact[o] * fact[e] if ok else 0

        used = [False] * (n + 1)
        o, e = (n + 1) // 2, n // 2
        res = []
        last = -1
        for _ in range(n):
            chosen = False
            for c in range(1, n + 1):
                if used[c] or c % 2 == last:
                    continue
                o2, e2 = (o - 1, e) if c % 2 else (o, e - 1)
                w = ways(o2, e2, c % 2 == 0)
                if k > w:
                    k -= w
                else:
                    used[c] = True
                    res.append(c)
                    o, e, last = o2, e2, c % 2
                    chosen = True
                    break
            if not chosen:
                return []
        return res

for n, k, exp in examples:
    got = Solution().permute(n, k)
    print(got, 'OK' if got == exp else 'SAI')

[3, 4, 1, 2] OK
[3, 2, 1] OK
[] OK


## 10. Vì sao thuật toán đúng

1. **Phân khối theo tiền tố:** các hoán vị xen kẽ theo thứ tự từ điển được chia thành các khối liên tiếp theo phần tử đầu tiên (tăng dần); trong mỗi khối lại chia theo phần tử thứ hai, … Hoán vị thứ `k` nằm trong khối đầu tiên mà tổng kích thước các khối tính tới nó ≥ `k`.
2. **Kích thước khối đúng:** khối ứng với tiền tố `P` (đã xen kẽ) gồm các cách xếp phần còn lại thành dãy xen kẽ bắt đầu bằng tính chẵn lẻ khác phần tử cuối của `P`. Mẫu chẵn lẻ của phần còn lại là **cố định**; nó khả thi ⇔ số lẻ/chẵn còn lại khớp mẫu (`o = e` hoặc lệch 1 đúng phía). Khi khả thi, số cách là `o! · e!` (xếp tự do số lẻ vào ô lẻ, số chẵn vào ô chẵn). Cách 2 xác nhận bằng đệ quy độc lập.
3. **Ứng viên không cùng tính với phần tử trước** bị loại vì vi phạm xen kẽ; ứng viên làm phần còn lại không khả thi có `w = 0` nên tự bị bỏ qua.
4. **`k` vượt tổng:** ở vị trí đầu, tổng các `w` là số hoán vị xen kẽ; nếu `k` lớn hơn thì vòng lặp không chọn được ai ⇒ `[]`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê | `O(n! · n)` | `O(n)` |
| 2. DP đếm | `O(n²)` + bảng nhớ `O(n²)` | `O(n²)` |
| 3. Công thức, trừ từng ứng viên | `O(n²)` phép tính trên số lớn | `O(n)` |
| 4. Nhảy bằng phép chia | `O(n²)` (do `list.remove`), có thể `O(n log n)` | `O(n)` |

Với `n = 100`: `O(n²) = 10^4` — tức thời. Số `50! · 50!` có ~130 chữ số, phép so sánh/trừ số lớn vẫn rất nhanh.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'dp': permute_dp, 'formula': permute_formula, 'jump': permute_jump,
         'submit': lambda n, k: Solution().permute(n, k)}
edge = [
    (1, 1, [1]), (1, 2, []),
    (2, 1, [1, 2]), (2, 2, [2, 1]),
    (3, 1, [1, 2, 3]), (3, 3, []),            # n lẻ: chỉ 2 hoán vị
    (5, 12, [5, 4, 3, 2, 1]), (5, 13, []),    # 3!·2! = 12
]
for n, k, exp in examples + edge:
    assert permute_brute(n, k) == exp, (n, k, permute_brute(n, k))
    for name, f in impls.items():
        assert f(n, k) == exp, (name, n, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu toàn bộ với liệt kê (n ≤ 8, mọi k tới tổng + 2)

In [13]:
cnt = 0
for n in range(1, 9):
    all_alt = [list(p) for p in itertools.permutations(range(1, n + 1)) if is_alternating(p)]
    for k in range(1, len(all_alt) + 3):
        exp = all_alt[k - 1] if k <= len(all_alt) else []
        for name, f in impls.items():
            assert f(n, k) == exp, (name, n, k)
        cnt += 1
print('Khớp liệt kê trên', cnt, 'cặp (n, k).')

Khớp liệt kê trên 1409 cặp (n, k).


### 12.3. Ca lớn: `n = 100`, `k` tới `10^15`; kiểm tra tính chất kết quả

In [14]:
import random, time
rng = random.Random(3470)
t = time.time()
for _ in range(300):
    n = rng.randint(50, 100)
    k = rng.randint(1, 10 ** 15)
    r = Solution().permute(n, k)
    assert sorted(r) == list(range(1, n + 1)) and is_alternating(r)
    assert r == permute_jump(n, k) == permute_dp(n, k)
print(f'300 bộ ngẫu nhiên n ∈ [50, 100], k ≤ 10^15: hợp lệ và 3 cách khớp nhau ({time.time() - t:.2f}s)')
print('n=100, k=10^15:', Solution().permute(100, 10 ** 15)[:12], '...')

# tính đơn điệu: k và k+1 cho hai hoán vị liên tiếp tăng dần
for _ in range(200):
    n = rng.randint(40, 100); k = rng.randint(1, 10 ** 15 - 1)      # n >= 40: tổng > 10^15
    assert Solution().permute(n, k) < Solution().permute(n, k + 1)
print('Hoán vị thứ k < hoán vị thứ k+1 (từ điển) trên 200 bộ.')

300 bộ ngẫu nhiên n ∈ [50, 100], k ≤ 10^15: hợp lệ và 3 cách khớp nhau (0.06s)
n=100, k=10^15: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] ...


Hoán vị thứ k < hoán vị thứ k+1 (từ điển) trên 200 bộ.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Luôn bắt đầu bằng số lẻ (kể cả khi `n` chẵn)

In [15]:
def wrong_start_odd(n, k):
    used = [False] * (n + 1)
    o, e = (n + 1) // 2, n // 2
    res, last = [], 0                       # SAI: coi như phần tử "trước" là chẵn
    for _ in range(n):
        for c in range(1, n + 1):
            if used[c] or c % 2 == last:
                continue
            o2, e2 = (o - 1, e) if c % 2 else (o, e - 1)
            w = ways(o2, e2, c % 2 == 0)
            if k > w:
                k -= w
            else:
                used[c] = True; res.append(c); o, e, last = o2, e2, c % 2
                break
        else:
            return []
    return res

print('sai :', wrong_start_odd(4, 3), '| đúng:', Solution().permute(4, 3))

sai : [3, 2, 1, 4] | đúng: [2, 1, 4, 3]


**Vì sao sai:** với `n` chẵn, mẫu C L C L … cũng hợp lệ (ví dụ `[2, 1, 4, 3]` là hoán vị thứ 3).

### Lỗi 2 — Bỏ qua điều kiện khả thi, luôn dùng `o! · e!`

In [16]:
def wrong_no_feasibility(n, k):
    used = [False] * (n + 1)
    o, e = (n + 1) // 2, n // 2
    res, last = [], None
    for _ in range(n):
        for c in range(1, n + 1):
            if used[c] or (last is not None and c % 2 == last):
                continue
            o2, e2 = (o - 1, e) if c % 2 else (o, e - 1)
            w = math.factorial(o2) * math.factorial(e2)     # SAI: không kiểm tra khả thi
            if k > w:
                k -= w
            else:
                used[c] = True; res.append(c); o, e, last = o2, e2, c % 2
                break
        else:
            return []
    return res

r = wrong_no_feasibility(3, 2)
print('sai :', r, '-> xen kẽ & đủ?', len(r) == 3 and is_alternating(r), '| đúng:', Solution().permute(3, 2))

sai : [] -> xen kẽ & đủ? False | đúng: [3, 2, 1]


**Vì sao sai:** với `n = 3`, bắt đầu bằng `2` thì còn hai số lẻ `1, 3` không thể đứng cạnh nhau ⇒ khối này rỗng. Công thức không kiểm tra khả thi đếm nó là `2` ⇒ chọn nhầm tiền tố và dựng ra dãy không hợp lệ (hoặc thiếu phần tử).

### Lỗi 3 — Đếm khối bằng `(o + e)!` (quên ràng buộc xen kẽ)

In [17]:
def wrong_all_perms(n, k):
    used = [False] * (n + 1)
    res, last = [], None
    for pos in range(n):
        for c in range(1, n + 1):
            if used[c] or (last is not None and c % 2 == last):
                continue
            w = math.factorial(n - pos - 1)                  # SAI
            if k > w:
                k -= w
            else:
                used[c] = True; res.append(c); last = c % 2
                break
        else:
            return []
    return res

print('sai :', wrong_all_perms(4, 6), '| đúng:', Solution().permute(4, 6))

sai : [] | đúng: [3, 4, 1, 2]


**Vì sao sai:** `(o + e)!` đếm cả những cách xếp vi phạm xen kẽ ⇒ khối bị phóng đại ⇒ chọn sai sớm.

### Lỗi 4 — Không xử lý `k` vượt quá tổng (trả về hoán vị cuối cùng)

In [18]:
def wrong_clamp(n, k):
    total = total_alternating(n)
    return Solution().permute(n, min(k, total))              # SAI: đề yêu cầu []

print('sai :', wrong_clamp(2, 3), '| đúng:', Solution().permute(2, 3))

sai : [2, 1] | đúng: []


**Vì sao sai:** đề yêu cầu trả `[]` khi không đủ `k` hoán vị.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đối tượng thứ `k` theo thứ tự từ điển" ⇒ dựng từng vị trí, **đếm số hoàn thành** cho mỗi ứng viên.
- Đếm hoàn thành thường có công thức đóng (giai thừa, tổ hợp) hoặc DP nhỏ.
- `k` tới `10^15` ⇒ ở C++/Java phải kẹp số đếm (`min(count, k + 1)`); Python không cần.
- Ràng buộc kề nhau chỉ theo tính chẵn lẻ ⇒ mẫu cố định, hai nhóm độc lập ⇒ `o! · e!`.

## 15. Biến thể của bài

1. **LC 60 — Permutation Sequence:** hoán vị thứ `k` không ràng buộc (`(n−1)!` mỗi khối).
2. Xen kẽ theo **số dư mod 3** (không hai phần tử kề nhau cùng dư) ⇒ cần đếm số mẫu dư hợp lệ (DP trên số lượng còn lại của từng lớp).
3. Đếm vị trí (thứ hạng) của một hoán vị xen kẽ cho trước — bài ngược.
4. Hoán vị thứ `k` với ràng buộc "không hai số liên tiếp đứng cạnh nhau" — khó hơn nhiều, cần DP bao hàm–loại trừ.

In [19]:
# Biến thể 3: thứ hạng (1-based) của một hoán vị xen kẽ cho trước
def rank_of(p):
    n = len(p)
    used = [False] * (n + 1)
    o, e = (n + 1) // 2, n // 2
    rank, last = 1, None
    for x in p:
        for c in range(1, x):
            if used[c] or (last is not None and c % 2 == last):
                continue
            o2, e2 = (o - 1, e) if c % 2 else (o, e - 1)
            rank += ways(o2, e2, c % 2 == 0)
        used[x] = True
        o, e = (o - 1, e) if x % 2 else (o, e - 1)
        last = x % 2
    return rank

for n in range(1, 8):
    all_alt = [list(p) for p in itertools.permutations(range(1, n + 1)) if is_alternating(p)]
    assert all(rank_of(p) == i for i, p in enumerate(all_alt, 1))
p = Solution().permute(100, 123456789012345)
print('rank_of(permute(100, 123456789012345)) =', rank_of(p))

rank_of(permute(100, 123456789012345)) = 123456789012345


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê | `O(n! n)` | Oracle |
| 2 | DP đếm hoàn thành | `O(n²)` | Không cần nhớ công thức |
| 3 | Công thức `o! e!` | `O(n²)` | **Nộp bài** |
| 4 | Nhảy bằng phép chia | `O(n²)` | Ít phép trừ hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 5, k = 1`.

**Bài 2 (tính tay):** `n = 4, k = 8`.

**Bài 3 (điền TODO):** viết `count_with_first(n, d)` — số hoán vị xen kẽ bắt đầu bằng `d`.

**Bài 4 (tự viết lại):** biến thể "giảm dần": hoán vị xen kẽ thứ `k` theo thứ tự từ điển **giảm dần**.

**Bài 5 (mini challenge):** biến thể 2 — không hai phần tử kề nhau có cùng số dư mod 3; trả về hoán vị thứ `k` (dùng DP đếm `(c0, c1, c2, last)`).

In [20]:
def count_with_first_todo(n, d):
    # TODO
    raise NotImplementedError

def permute_desc_todo(n, k):
    # TODO
    raise NotImplementedError

def permute_mod3_todo(n, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Thứ tự từ điển nhỏ nhất. (2) `n` lẻ ⇒ bắt đầu lẻ. (3) `[1, 2, 3, 4, 5]` có xen kẽ không?

**Bài 2.** (1) Tổng `2 · 2! · 2! = 8`. (2) `k = 8` là hoán vị **cuối**. (3) Là hoán vị xen kẽ lớn nhất theo từ điển.

**Bài 3.** (1) Bỏ `d`, còn `(o', e')`. (2) Kế tiếp phải khác tính của `d`. (3) `ways(o', e', d chẵn)`; nhớ kiểm tra `n` lẻ mà `d` chẵn.

**Bài 4.** (1) Duyệt ứng viên từ lớn xuống nhỏ. (2) Phần đếm không đổi. (3) Hoặc: hoán vị thứ `total − k + 1` theo thứ tự tăng.

**Bài 5.** (1) Trạng thái `(số còn lại mỗi lớp dư, lớp của phần tử cuối)`. (2) Số cách = số mẫu lớp × tích giai thừa các lớp. (3) Đối chiếu liệt kê với `n ≤ 8`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
a1 = Solution().permute(5, 1)
a2 = Solution().permute(4, 8)
assert a1 == permute_brute(5, 1) == [1, 2, 3, 4, 5]
assert a2 == permute_brute(4, 8) == [4, 3, 2, 1]
print('Bài 1:', a1, '| Bài 2:', a2)

def count_with_first(n, d):
    o, e = (n + 1) // 2, n // 2
    o2, e2 = (o - 1, e) if d % 2 else (o, e - 1)
    return ways(o2, e2, d % 2 == 0)

for n in range(1, 8):
    for d in range(1, n + 1):
        exp = sum(1 for p in itertools.permutations(range(1, n + 1)) if p[0] == d and is_alternating(p))
        assert count_with_first(n, d) == exp
print('Bài 3 khớp liệt kê; n=6:', [count_with_first(6, d) for d in range(1, 7)])

Bài 1: [1, 2, 3, 4, 5] | Bài 2: [4, 3, 2, 1]
Bài 3 khớp liệt kê; n=6: [12, 12, 12, 12, 12, 12]


In [22]:
# Bài 4
def permute_desc(n, k):
    total = total_alternating(n)
    if k > total:
        return []
    return Solution().permute(n, total - k + 1)

for n in range(1, 8):
    all_alt = sorted((list(p) for p in itertools.permutations(range(1, n + 1)) if is_alternating(p)), reverse=True)
    for k in range(1, len(all_alt) + 2):
        assert permute_desc(n, k) == (all_alt[k - 1] if k <= len(all_alt) else [])
print('Bài 4 khớp liệt kê:', permute_desc(4, 1), permute_desc(4, 2))

Bài 4 khớp liệt kê: [4, 3, 2, 1] [4, 1, 2, 3]


In [23]:
# Bài 5 — mod 3
def permute_mod3(n, k):
    from math import factorial
    cls = [[x for x in range(1, n + 1) if x % 3 == r] for r in range(3)]

    @lru_cache(maxsize=None)
    def patterns(c0, c1, c2, last):          # số mẫu lớp dư hợp lệ cho phần còn lại
        cnt = (c0, c1, c2)
        if sum(cnt) == 0:
            return 1
        total = 0
        for r in range(3):
            if r != last and cnt[r]:
                nxt = list(cnt); nxt[r] -= 1
                total += patterns(*nxt, r)
        return total

    def completions(cnt, last):
        return patterns(*cnt, last) * factorial(cnt[0]) * factorial(cnt[1]) * factorial(cnt[2])

    cnt = [len(c) for c in cls]
    used = [False] * (n + 1)
    res, last = [], -1
    for _ in range(n):
        for c in range(1, n + 1):
            if used[c] or c % 3 == last:
                continue
            nxt = cnt[:]; nxt[c % 3] -= 1
            w = completions(tuple(nxt), c % 3)
            if k > w:
                k -= w
            else:
                used[c] = True; res.append(c); cnt = nxt; last = c % 3
                break
        else:
            return []
    return res

for n in range(1, 8):
    good = [list(p) for p in itertools.permutations(range(1, n + 1)) if all(p[i] % 3 != p[i + 1] % 3 for i in range(n - 1))]
    for k in range(1, len(good) + 2):
        assert permute_mod3(n, k) == (good[k - 1] if k <= len(good) else [])
print('Bài 5 khớp liệt kê với n ≤ 7. Ví dụ n=9, k=1000:', permute_mod3(9, 1000))

Bài 5 khớp liệt kê với n ≤ 7. Ví dụ n=9, k=1000: [1, 3, 5, 4, 9, 7, 8, 6, 2]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ways(o, e, need_odd) = o!·e! nếu (need_odd: o in {e, e+1}; ngược lại: e in {o, o+1}), else 0
for mỗi vị trí:
    for c tăng dần, chưa dùng, khác tính chẵn lẻ phần tử trước:
        w = ways(o', e', c chẵn)      # (o', e') sau khi dùng c
        if k > w: k -= w
        else: chọn c; break
    nếu không chọn được -> []
```

**Checklist:**

- [ ] `n` chẵn: vị trí đầu cho phép cả lẻ và chẵn?
- [ ] Kiểm tra khả thi trước khi dùng `o! · e!`?
- [ ] `k` vượt tổng ⇒ `[]`?
- [ ] Ngôn ngữ khác: kẹp số đếm để không tràn?
- [ ] Hết phần tử (`o + e = 0`) ⇒ 1 cách?